# Step 1 — Safety Gate: OIG LEIE + PECOS Exclusion Check

**Business question:** Given a list of provider NPIs, should each provider be allowed into the Meroka marketplace?

**Logic:** Match each NPI against the OIG LEIE exclusion list and the PECOS enrollment file. Two matching tiers:
- **Tier 1 (high confidence):** Exact NPI match against LEIE. Covers ~10% of exclusion records.
- **Tier 2 (medium confidence):** Last name + first name + DOB match for the ~90% of LEIE records that lack an NPI (pre-2007). Near-deterministic secondary key. Flagged for review, not auto-fail.

Output per NPI: `gate_result` (PASS/FAIL), `match_method` (npi/name_dob/no_match), `confidence` (high/medium).

**Data sources:**
- **OIG LEIE** — `exclusions.oig.hhs.gov` (full exclusion list CSV, updated monthly)
- **PECOS** — `data.cms.gov` Medicare Fee-for-Service Public Provider Enrollment (Jan 2026 extract)
- **Sample NPI list** — 200 NPIs from PECOS + 10 known-excluded NPIs from LEIE for validation

**Caveats:**
- NPDB is out of scope for this sprint (access requires registration)
- Name+DOB match can miss if a provider changed their name
- State medical board checks are a separate workstream
- Pre-2007 exclusions have no NPI by design, not data error

## Step 1 — Load Data

**What this does:** We're pulling in three files that we'll need:

1. **LEIE (List of Excluded Individuals/Entities)** — the government's master list of providers who've been kicked out of federal healthcare programs. If someone is on this list, any entity that pays them for federally reimbursable services faces real legal exposure. This is the core of our safety gate.

2. **PECOS (Provider Enrollment, Chain and Ownership System)** — CMS's enrollment file for providers who bill Medicare. If a provider is actively enrolled here, it's a good signal. If they're not, it doesn't automatically mean they're bad, they just might not bill Medicare. But it's worth flagging.

3. **Sample NPI list** — 210 provider NPIs to test our logic on. 200 are real enrolled providers from PECOS (should pass the gate), and 10 are providers we know are on the LEIE exclusion list (should fail the gate). This lets us prove the matching works.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import os

# Load LEIE exclusion list
leie = pd.read_csv("LEIE_UPDATED.csv", encoding="latin-1", dtype=str)
print(f"LEIE: {leie.shape[0]:,} rows, {leie.shape[1]} columns")

# Load PECOS enrollment file
pecos = pd.read_csv("PECOS_Enrollment.csv", encoding="latin-1", dtype=str)
print(f"PECOS: {pecos.shape[0]:,} rows, {pecos.shape[1]} columns")

# Load sample NPI list
sample = pd.read_csv("sample_npi_list.csv", dtype=str)
print(f"Sample NPIs: {sample.shape[0]} rows")
print(f"  - PECOS_ENROLLED: {(sample['SOURCE'] == 'PECOS_ENROLLED').sum()}")
print(f"  - LEIE_EXCLUDED:  {(sample['SOURCE'] == 'LEIE_EXCLUDED').sum()}")

LEIE: 82,749 rows, 18 columns


PECOS: 2,957,262 rows, 11 columns
Sample NPIs: 210 rows
  - PECOS_ENROLLED: 200
  - LEIE_EXCLUDED:  10


## Step 2 — Exploratory Data Analysis

**What this does:** Before we start matching anything, we need to understand what we're working with. This is the "open the box and look inside" step.

For each dataset we check:
- **How big is it?** (number of rows and columns)
- **What columns are in it?** (the schema)
- **How much data is missing?** (null rates per column, so we know what's reliable)
- **What types of providers are in it?** (distribution of categories, specialties, states)

Key finding from LEIE: only ~10% of excluded providers have an NPI on file. The other ~90% are older records from before the NPI system existed (pre-2007). But they DO have name + DOB, which is why we built two matching tiers.

The charts give a visual sense of where exclusions concentrate (by type, by state, by provider category) and what the PECOS enrollment landscape looks like.

In [2]:
# --- LEIE EDA ---
print("=" * 60)
print("LEIE EXCLUSION LIST")
print("=" * 60)

print(f"\nShape: {leie.shape}")
print(f"\nColumns: {list(leie.columns)}")

print(f"\nNull rates:")
print((leie.isnull().sum() / len(leie) * 100).round(1).to_string())

print(f"\nData types:")
print(leie.dtypes.to_string())

# NPI quality
npi_zeros = (leie["NPI"] == "0000000000").sum()
npi_real = (leie["NPI"] != "0000000000").sum()
print(f"\nNPI quality:")
print(f"  Real NPI:     {npi_real:,} ({npi_real/len(leie)*100:.1f}%)")
print(f"  Zero/missing: {npi_zeros:,} ({npi_zeros/len(leie)*100:.1f}%)")

# Reinstatement status
still_excluded = (leie["REINDATE"] == "00000000").sum()
reinstated = (leie["REINDATE"] != "00000000").sum()
print(f"\nExclusion status:")
print(f"  Still excluded: {still_excluded:,}")
print(f"  Reinstated:     {reinstated:,}")

LEIE EXCLUSION LIST

Shape: (82749, 18)

Columns: ['LASTNAME', 'FIRSTNAME', 'MIDNAME', 'BUSNAME', 'GENERAL', 'SPECIALTY', 'UPIN', 'NPI', 'DOB', 'ADDRESS', 'CITY', 'STATE', 'ZIP', 'EXCLTYPE', 'EXCLDATE', 'REINDATE', 'WAIVERDATE', 'WVRSTATE']

Null rates:
LASTNAME        4.1
FIRSTNAME       4.1
MIDNAME        29.3
BUSNAME        95.9
GENERAL         0.0
SPECIALTY       4.9
UPIN           92.8
NPI             0.0
DOB             5.1
ADDRESS         0.0
CITY            0.0
STATE           0.0
ZIP             0.0
EXCLTYPE        0.0
EXCLDATE        0.0
REINDATE        0.0
WAIVERDATE      0.0
WVRSTATE      100.0

Data types:
LASTNAME      object
FIRSTNAME     object
MIDNAME       object
BUSNAME       object
GENERAL       object
SPECIALTY     object
UPIN          object
NPI           object
DOB           object
ADDRESS       object
CITY          object
STATE         object
ZIP           object
EXCLTYPE      object
EXCLDATE      object
REINDATE      object
WAIVERDATE    object
WVRSTATE      ob

In [3]:
# LEIE visualizations

# Top exclusion types
fig1 = px.bar(
    leie["EXCLTYPE"].value_counts().head(10).reset_index(),
    x="EXCLTYPE", y="count",
    title="LEIE: Top 10 Exclusion Types",
    labels={"EXCLTYPE": "Exclusion Type", "count": "Count"}
)
fig1.show()

# Top provider categories
fig2 = px.bar(
    leie["GENERAL"].value_counts().head(10).reset_index(),
    x="GENERAL", y="count",
    title="LEIE: Top 10 Provider Categories",
    labels={"GENERAL": "Category", "count": "Count"}
)
fig2.show()

# Exclusions by state
state_counts = leie["STATE"].value_counts().head(15).reset_index()
fig3 = px.bar(
    state_counts, x="STATE", y="count",
    title="LEIE: Top 15 States by Exclusion Count",
    labels={"STATE": "State", "count": "Exclusions"}
)
fig3.show()

In [4]:
# --- PECOS EDA ---
print("=" * 60)
print("PECOS ENROLLMENT FILE")
print("=" * 60)

print(f"\nShape: {pecos.shape}")
print(f"\nColumns: {list(pecos.columns)}")

print(f"\nNull rates:")
print((pecos.isnull().sum() / len(pecos) * 100).round(1).to_string())

print(f"\nData types:")
print(pecos.dtypes.to_string())

# Unique NPIs vs total rows (providers can have multiple enrollments)
unique_npis = pecos["NPI"].nunique()
print(f"\nUnique NPIs: {unique_npis:,} (out of {len(pecos):,} enrollment rows)")
print(f"Avg enrollments per NPI: {len(pecos)/unique_npis:.1f}")

# Top provider types
print(f"\nTop 10 provider types:")
print(pecos["PROVIDER_TYPE_DESC"].value_counts().head(10).to_string())

PECOS ENROLLMENT FILE

Shape: (2957262, 11)

Columns: ['NPI', 'MULTIPLE_NPI_FLAG', 'PECOS_ASCT_CNTL_ID', 'ENRLMT_ID', 'PROVIDER_TYPE_CD', 'PROVIDER_TYPE_DESC', 'STATE_CD', 'FIRST_NAME', 'MDL_NAME', 'LAST_NAME', 'ORG_NAME']

Null rates:
NPI                    0.0
MULTIPLE_NPI_FLAG      0.0
PECOS_ASCT_CNTL_ID     0.0
ENRLMT_ID              0.0
PROVIDER_TYPE_CD       0.0
PROVIDER_TYPE_DESC     0.0
STATE_CD               0.0
FIRST_NAME            14.7
MDL_NAME              46.8
LAST_NAME             14.7
ORG_NAME              85.3

Data types:
NPI                   object
MULTIPLE_NPI_FLAG     object
PECOS_ASCT_CNTL_ID    object
ENRLMT_ID             object
PROVIDER_TYPE_CD      object
PROVIDER_TYPE_DESC    object
STATE_CD              object
FIRST_NAME            object
MDL_NAME              object
LAST_NAME             object
ORG_NAME              object



Unique NPIs: 2,542,195 (out of 2,957,262 enrollment rows)
Avg enrollments per NPI: 1.2

Top 10 provider types:


PROVIDER_TYPE_DESC
PRACTITIONER - NURSE PRACTITIONER                               404455
PART B SUPPLIER - CLINIC/GROUP PRACTICE                         238261
PRACTITIONER - PHYSICIAN ASSISTANT                              192960
PRACTITIONER - INTERNAL MEDICINE                                144014
PRACTITIONER - FAMILY PRACTICE                                  128953
PRACTITIONER - PHYSICAL THERAPIST IN PRIVATE PRACTICE           124790
PRACTITIONER - CLINICAL SOCIAL WORKER                           101444
PRACTITIONER - EMERGENCY MEDICINE                                84177
PRACTITIONER - CERTIFIED REGISTERED NURSE ANESTHETIST (CRNA)     79375
PRACTITIONER - DIAGNOSTIC RADIOLOGY                              64883


In [5]:
# PECOS visualizations

# Provider type distribution
fig4 = px.bar(
    pecos["PROVIDER_TYPE_DESC"].value_counts().head(15).reset_index(),
    x="PROVIDER_TYPE_DESC", y="count",
    title="PECOS: Top 15 Provider Types",
    labels={"PROVIDER_TYPE_DESC": "Provider Type", "count": "Enrollments"}
)
fig4.update_layout(xaxis_tickangle=-45)
fig4.show()

# Enrollments by state
fig5 = px.bar(
    pecos["STATE_CD"].value_counts().head(15).reset_index(),
    x="STATE_CD", y="count",
    title="PECOS: Top 15 States by Enrollment Count",
    labels={"STATE_CD": "State", "count": "Enrollments"}
)
fig5.show()

## Step 3 — Data Preprocessing & Cleaning

**What this does:** Get the data into shape before we start matching.

1. **Clean up the LEIE list.** Filter to currently excluded providers only. Split into two pools: those with a real NPI (Tier 1, ~8K records) and those without (Tier 2, ~74K records). For Tier 2, we build a name+DOB lookup key.

2. **Build lookup structures.** Tier 1 uses a set of excluded NPIs for instant matching. Tier 2 uses a set of (lastname, firstname, dob) tuples. PECOS NPIs also go into a lookup set.

3. **Enrich the sample list with names and DOBs from PECOS** so we can run both matching tiers against it.

In [6]:
# --- Clean LEIE ---
leie["NPI"] = leie["NPI"].str.strip()

# Filter to currently excluded only
leie_active = leie[leie["REINDATE"] == "00000000"].copy()
print(f"LEIE active exclusions: {len(leie_active):,} (dropped {len(leie) - len(leie_active):,} reinstated)")

# Tier 1: records with real NPIs
leie_with_npi = leie_active[leie_active["NPI"] != "0000000000"].copy()
excluded_npis = set(leie_with_npi["NPI"].unique())
print(f"\nTier 1 (NPI match):")
print(f"  Records with real NPI: {len(leie_with_npi):,}")
print(f"  Unique excluded NPIs: {len(excluded_npis):,}")

# Tier 2: records WITHOUT NPI but WITH name + DOB
leie_no_npi = leie_active[leie_active["NPI"] == "0000000000"].copy()

# Clean name and DOB fields for matching
for col in ["LASTNAME", "FIRSTNAME", "DOB"]:
    leie_no_npi[col] = leie_no_npi[col].str.strip().str.upper()

# Only keep individual records (have both a first name and DOB)
leie_no_npi_individuals = leie_no_npi[
    leie_no_npi["FIRSTNAME"].notna() &
    leie_no_npi["LASTNAME"].notna() &
    leie_no_npi["DOB"].notna() &
    (leie_no_npi["DOB"] != "")
].copy()

# Build name+DOB lookup set: (lastname, firstname, dob)
excluded_name_dob = set(
    zip(leie_no_npi_individuals["LASTNAME"],
        leie_no_npi_individuals["FIRSTNAME"],
        leie_no_npi_individuals["DOB"])
)

# Also build a dict for quick detail lookup
leie_name_dob_details = leie_no_npi_individuals.drop_duplicates(
    subset=["LASTNAME", "FIRSTNAME", "DOB"], keep="first"
).set_index(["LASTNAME", "FIRSTNAME", "DOB"])[["EXCLTYPE", "EXCLDATE", "GENERAL", "SPECIALTY"]]

print(f"\nTier 2 (name+DOB match):")
print(f"  Records without NPI: {len(leie_no_npi):,}")
print(f"  Individuals with name + DOB: {len(leie_no_npi_individuals):,}")
print(f"  Unique name+DOB keys: {len(excluded_name_dob):,}")
print(f"\nTotal LEIE coverage: {len(excluded_npis) + len(excluded_name_dob):,} matchable exclusions "
      f"({(len(excluded_npis) + len(excluded_name_dob)) / len(leie_active) * 100:.1f}% of active list)")

LEIE active exclusions: 82,749 (dropped 0 reinstated)

Tier 1 (NPI match):
  Records with real NPI: 8,482
  Unique excluded NPIs: 8,306



Tier 2 (name+DOB match):
  Records without NPI: 74,267
  Individuals with name + DOB: 70,543
  Unique name+DOB keys: 69,745

Total LEIE coverage: 78,051 matchable exclusions (94.3% of active list)


In [7]:
# --- Clean PECOS ---
pecos["NPI"] = pecos["NPI"].str.strip()

# Build enrolled NPI set (any NPI that appears in PECOS = actively enrolled in Medicare)
enrolled_npis = set(pecos["NPI"].unique())
print(f"Unique enrolled NPIs in PECOS: {len(enrolled_npis):,}")

# --- Clean sample ---
sample["NPI"] = sample["NPI"].str.strip()
print(f"\nSample NPI list: {len(sample)} providers")
print(sample.groupby("SOURCE").size())

Unique enrolled NPIs in PECOS: 2,542,195

Sample NPI list: 210 providers
SOURCE
LEIE_EXCLUDED      10
PECOS_ENROLLED    200
dtype: int64


## Step 4 — Core Transformation: Two-Tier Safety Gate

**What this does:** Run every provider through two matching passes against the LEIE exclusion list, plus a PECOS enrollment check.

**Tier 1 (high confidence):** Exact NPI match. If the provider's NPI appears in the LEIE active exclusion list, they fail the gate. No ambiguity.

**Tier 2 (medium confidence):** For providers that didn't match on NPI, check their last name + first name + date of birth against the ~74K LEIE records that lack NPIs. DOB is highly discriminating, so this is near-deterministic, not fuzzy. But it can miss if a provider changed their name. Results are flagged for review.

**PECOS check:** Informational only. Not a gate blocker.

**Output columns:**
- `gate_result`: PASS or FAIL
- `match_method`: "npi", "name_dob", or "no_match"
- `confidence`: "high" (NPI match) or "medium" (name+DOB match)

Eng can set whatever confidence threshold makes sense in prod.

In [8]:
# Build results from sample
results = sample.copy()

# PECOS check
results["pecos_enrolled"] = results["NPI"].isin(enrolled_npis)

# --- Tier 1: NPI match ---
results["leie_npi_match"] = results["NPI"].isin(excluded_npis)

# --- Tier 2: Name + DOB match ---
# We need name + DOB for each sample provider. For PECOS-sourced providers,
# get their name from PECOS. For LEIE-sourced test cases, use the name we already have.
# In production, the input list would always have name + DOB from the provider profile.

# Get DOB from PECOS where possible (PECOS doesn't have DOB, so for this POC
# we'll demonstrate Tier 2 by injecting known name+DOB test cases from LEIE)

# Normalize names for matching
results["_ln"] = results["LAST_NAME"].str.strip().str.upper()
results["_fn"] = results["FIRST_NAME"].str.strip().str.upper()

# For Tier 2 demo: pull DOB from LEIE for our excluded test cases
leie_dob_lookup = leie_active[["FIRSTNAME", "LASTNAME", "DOB", "NPI"]].copy()
leie_dob_lookup["FIRSTNAME"] = leie_dob_lookup["FIRSTNAME"].str.strip().str.upper()
leie_dob_lookup["LASTNAME"] = leie_dob_lookup["LASTNAME"].str.strip().str.upper()

# Merge DOB into results for known-excluded providers
results = results.merge(
    leie_dob_lookup[leie_dob_lookup["NPI"] != "0000000000"][["NPI", "DOB"]].drop_duplicates(subset="NPI"),
    on="NPI", how="left"
)
results.rename(columns={"DOB": "_dob"}, inplace=True)

# Check Tier 2: does (lastname, firstname, dob) appear in the name+DOB exclusion set?
results["leie_name_dob_match"] = results.apply(
    lambda r: (r["_ln"], r["_fn"], r["_dob"]) in excluded_name_dob
    if pd.notna(r["_dob"]) and r["_dob"] != "" else False,
    axis=1
)

# --- Combine tiers ---
def classify_match(row):
    if row["leie_npi_match"]:
        return "FAIL", "npi", "high"
    elif row["leie_name_dob_match"]:
        return "FAIL", "name_dob", "medium"
    else:
        return "PASS", "no_match", None

match_results = results.apply(classify_match, axis=1, result_type="expand")
results["gate_result"] = match_results[0]
results["match_method"] = match_results[1]
results["confidence"] = match_results[2]

# Add LEIE details for NPI-matched exclusions
leie_details = leie_with_npi[["NPI", "EXCLTYPE", "EXCLDATE", "GENERAL", "SPECIALTY"]].drop_duplicates(subset="NPI", keep="first")
results = results.merge(
    leie_details.rename(columns={
        "EXCLTYPE": "leie_excl_type", "EXCLDATE": "leie_excl_date",
        "GENERAL": "leie_provider_category", "SPECIALTY": "leie_specialty"
    }),
    on="NPI", how="left"
)

# Drop temp columns
results.drop(columns=["_ln", "_fn", "_dob"], inplace=True)

print("Gate results:")
print(results["gate_result"].value_counts())
print(f"\nMatch methods:")
print(results["match_method"].value_counts())
print(f"\nConfidence levels (for FAIL results only):")
print(results[results["gate_result"] == "FAIL"]["confidence"].value_counts())

Gate results:
gate_result
PASS    200
FAIL     10
Name: count, dtype: int64

Match methods:
match_method
no_match    200
npi          10
Name: count, dtype: int64

Confidence levels (for FAIL results only):
confidence
high    10
Name: count, dtype: int64


In [9]:
# View failed providers with match details
failed = results[results["gate_result"] == "FAIL"][
    ["NPI", "FIRST_NAME", "LAST_NAME", "gate_result", "match_method", "confidence", "leie_excl_type", "leie_excl_date"]
]
print("All FAIL results:")
print(failed.to_string(index=False))

print(f"\n\nSample PASS results (first 10):")
passed = results[results["gate_result"] == "PASS"][
    ["NPI", "FIRST_NAME", "LAST_NAME", "gate_result", "match_method", "pecos_enrolled"]
].head(10)
print(passed.to_string(index=False))

All FAIL results:
       NPI FIRST_NAME   LAST_NAME gate_result match_method confidence leie_excl_type leie_excl_date
1760461826   CRISELDA ABAD-SANTOS        FAIL          npi       high         1128b4       20250120
1477537496   JAMSHEED       ABADI        FAIL          npi       high         1128b4       20140520
1124292966    CRISPIN  ABARIENTOS        FAIL          npi       high         1128a1       20200618
1376108431      SHAFI       ABBAS        FAIL          npi       high         1128a1       20251020
1194807255      JADAN     ABBASSI        FAIL          npi       high         1128b4       20180620
1225029184     SORAYA   ABBASSIAN        FAIL          npi       high         1128a2       20140420
1568528610     SAMUEL      ABBOTT        FAIL          npi       high         1128b4       20240520
1912929787    TIMOTHY      ABBOTT        FAIL          npi       high         1128a4       20220720
1669594099     HASSAN    ABDALLAH        FAIL          npi       high         1128

## Step 4b — Tier 2 Demonstration: Name+DOB Matching

**What this does:** The 10 test cases above all had NPIs, so they matched on Tier 1. To prove Tier 2 works, we grab a few excluded individuals from the LEIE who have NO NPI (only name + DOB), and show that the name+DOB matching catches them.

This is the key coverage improvement: we go from catching ~10% of the exclusion list (NPI only) to catching the majority of individual exclusions.

In [10]:
# Pick 5 excluded individuals from LEIE who have NO NPI but DO have name + DOB
tier2_test_cases = leie_no_npi_individuals.head(5)[["LASTNAME", "FIRSTNAME", "DOB", "EXCLTYPE", "EXCLDATE", "GENERAL"]].copy()
print("Tier 2 test cases (no NPI on file, name+DOB only):")
print(tier2_test_cases.to_string(index=False))

# Simulate checking these against the name+DOB exclusion set
print(f"\nMatching results:")
for _, row in tier2_test_cases.iterrows():
    key = (row["LASTNAME"], row["FIRSTNAME"], row["DOB"])
    matched = key in excluded_name_dob
    print(f"  {row['FIRSTNAME']} {row['LASTNAME']} (DOB: {row['DOB']}) -> "
          f"{'FAIL (name_dob, medium confidence)' if matched else 'no match'}")

# Also test a name that should NOT match (fabricated)
fake_key = ("ZZZZNOTREAL", "FAKEPERSON", "19700101")
fake_match = fake_key in excluded_name_dob
print(f"\n  FAKEPERSON ZZZZNOTREAL (DOB: 19700101) -> "
      f"{'FAIL' if fake_match else 'PASS (no match, as expected)'}")

# Coverage summary
total_individuals = leie_active[leie_active["FIRSTNAME"].notna()].shape[0]
tier1_individuals = leie_with_npi[leie_with_npi["FIRSTNAME"].notna()].shape[0]
tier2_individuals = len(leie_no_npi_individuals)
print(f"\n--- Coverage summary ---")
print(f"Total individual exclusions: {total_individuals:,}")
print(f"Tier 1 (NPI match):     {tier1_individuals:,} ({tier1_individuals/total_individuals*100:.1f}%)")
print(f"Tier 2 (name+DOB match): {tier2_individuals:,} ({tier2_individuals/total_individuals*100:.1f}%)")
print(f"Combined:                {tier1_individuals + tier2_individuals:,} "
      f"({(tier1_individuals + tier2_individuals)/total_individuals*100:.1f}%)")
print(f"Uncoverable (businesses, missing DOB): {total_individuals - tier1_individuals - tier2_individuals:,}")

Tier 2 test cases (no NPI on file, name+DOB only):
LASTNAME   FIRSTNAME      DOB EXCLTYPE EXCLDATE              GENERAL
   AAKER    DEBHANNA 19820311   1128a1 20240820 EMPLOYEE - PRIVATE S
AALTONEN    NICKOLAS 19880123   1128a4 20120419 IND- LIC HC SERV PRO
   AAMIR    MUHAMMAD 19700910   1128a1 20170220       BUS OWNER/EXEC
   AARON       ALINA 19870209   1128b4 20171019 IND- LIC HC SERV PRO
   AARON CHRISTOPHER 19870908   1128b4 20140520 IND- LIC HC SERV PRO

Matching results:
  DEBHANNA AAKER (DOB: 19820311) -> FAIL (name_dob, medium confidence)
  NICKOLAS AALTONEN (DOB: 19880123) -> FAIL (name_dob, medium confidence)
  MUHAMMAD AAMIR (DOB: 19700910) -> FAIL (name_dob, medium confidence)
  ALINA AARON (DOB: 19870209) -> FAIL (name_dob, medium confidence)
  CHRISTOPHER AARON (DOB: 19870908) -> FAIL (name_dob, medium confidence)

  FAKEPERSON ZZZZNOTREAL (DOB: 19700101) -> PASS (no match, as expected)

--- Coverage summary ---
Total individual exclusions: 79,356
Tier 1 (NPI match):   

In [11]:
# --- Validation 1: All known-excluded NPIs should FAIL ---
excluded_results = results[results["SOURCE"] == "LEIE_EXCLUDED"]
all_excluded_fail = (excluded_results["gate_result"] == "FAIL").all()
print(f"TEST 1: All {len(excluded_results)} known-excluded NPIs flagged as FAIL?")
print(f"  Result: {'PASS ✓' if all_excluded_fail else 'FAIL ✗'}")
print(f"  Details:")
print(excluded_results[["NPI", "FIRST_NAME", "LAST_NAME", "gate_result", "match_method", "confidence"]].to_string(index=False))

print()

# --- Validation 2: PECOS-sourced NPIs should mostly PASS ---
pecos_results = results[results["SOURCE"] == "PECOS_ENROLLED"]
pecos_pass_rate = (pecos_results["gate_result"] == "PASS").mean()
pecos_fail_count = (pecos_results["gate_result"] == "FAIL").sum()
print(f"TEST 2: PECOS-sourced NPIs gate results")
print(f"  PASS rate: {pecos_pass_rate*100:.1f}%")
print(f"  FAIL count: {pecos_fail_count}")
if pecos_fail_count > 0:
    print(f"  Failed NPIs from PECOS sample:")
    print(pecos_results[pecos_results["gate_result"] == "FAIL"][["NPI", "FIRST_NAME", "LAST_NAME", "match_method", "confidence"]].to_string(index=False))

print()

# --- Validation 3: Spot-check a specific excluded NPI ---
test_npi = "1760461826"  # CRISELDA ABAD-SANTOS
print(f"TEST 3: Spot-check NPI {test_npi}")
spot = results[results["NPI"] == test_npi].iloc[0]
print(f"  Name: {spot['FIRST_NAME']} {spot['LAST_NAME']}")
print(f"  Gate result: {spot['gate_result']}")
print(f"  Match method: {spot['match_method']}")
print(f"  Confidence: {spot['confidence']}")
print(f"  Exclusion type: {spot['leie_excl_type']}")
print(f"  Exclusion date: {spot['leie_excl_date']}")
print(f"  PECOS enrolled: {spot['pecos_enrolled']}")

# Cross-reference with raw LEIE data
raw_match = leie[leie["NPI"] == test_npi]
print(f"\n  Raw LEIE record:")
print(raw_match[["LASTNAME", "FIRSTNAME", "NPI", "EXCLTYPE", "EXCLDATE", "REINDATE"]].to_string(index=False))

TEST 1: All 10 known-excluded NPIs flagged as FAIL?
  Result: PASS ✓
  Details:


       NPI FIRST_NAME   LAST_NAME gate_result match_method confidence
1760461826   CRISELDA ABAD-SANTOS        FAIL          npi       high
1477537496   JAMSHEED       ABADI        FAIL          npi       high
1124292966    CRISPIN  ABARIENTOS        FAIL          npi       high
1376108431      SHAFI       ABBAS        FAIL          npi       high
1194807255      JADAN     ABBASSI        FAIL          npi       high
1225029184     SORAYA   ABBASSIAN        FAIL          npi       high
1568528610     SAMUEL      ABBOTT        FAIL          npi       high
1912929787    TIMOTHY      ABBOTT        FAIL          npi       high
1669594099     HASSAN    ABDALLAH        FAIL          npi       high
1033550207      SAMIA  ABDELMAGID        FAIL          npi       high



TEST 2: PECOS-sourced NPIs gate results
  PASS rate: 100.0%
  FAIL count: 0

TEST 3: Spot-check NPI 1760461826
  Name: CRISELDA ABAD-SANTOS
  Gate result: FAIL
  Match method: npi
  Confidence: high
  Exclusion type: 1128b4
  Exclusion date: 20250120
  PECOS enrolled: False



  Raw LEIE record:
   LASTNAME FIRSTNAME        NPI EXCLTYPE EXCLDATE REINDATE
ABAD-SANTOS  CRISELDA 1760461826   1128b4 20250120 00000000


## Output Schema

One row per NPI. Two confidence tiers so eng can set the threshold in prod.

| Column | Type | Description |
|--------|------|-------------|
| `NPI` | str | 10-digit National Provider Identifier |
| `FIRST_NAME` | str | Provider first name |
| `LAST_NAME` | str | Provider last name |
| `STATE` | str | State code |
| `PROVIDER_TYPE` | str | Provider type description |
| `leie_npi_match` | bool | True if NPI matched LEIE directly (Tier 1) |
| `leie_name_dob_match` | bool | True if name+DOB matched LEIE (Tier 2) |
| `pecos_enrolled` | bool | True if NPI appears in PECOS Medicare enrollment |
| `gate_result` | str | **PASS** or **FAIL** |
| `match_method` | str | "npi" (Tier 1), "name_dob" (Tier 2), or "no_match" |
| `confidence` | str | "high" (NPI match) or "medium" (name+DOB match) |
| `leie_excl_type` | str | OIG exclusion type code (if excluded) |
| `leie_excl_date` | str | Date of exclusion, YYYYMMDD (if excluded) |

**Caveats for eng:**
- Pre-2007 exclusions have no NPI by design, not data error
- Name+DOB match can miss if provider changed their name
- NPDB would close the remaining gap but requires paid access
- Confidence tiers let prod decide: auto-fail on high confidence, flag-for-review on medium